# Respostas das perguntas

Este notebook utiliza as tabelas da camada Gold (`fato_obitos`, `dim_localidade`,
`dim_causa`) para responder às quatro perguntas definidas no início do projeto,
por meio de consultas SQL.

In [0]:
%sql
-- Total de óbitos em Salvador (2020-2025)
SELECT
  COUNT(*) AS total_obitos
FROM mvp_engenharia_dados.gold.fato_obitos f
JOIN mvp_engenharia_dados.gold.dim_localidade l
  ON f.ID_MUNICIPIO_RESIDENCIA = l.ID_MUNICIPIO
WHERE l.FLAG_SALVADOR = true

In [0]:
%sql
-- Principais causas de óbito em Salvador
SELECT
  c.CODIGO_CID,
  c.NOME_CID,
  c.NOME_CAPITULO,
  COUNT(*) AS total_obitos
FROM mvp_engenharia_dados.gold.fato_obitos f
JOIN mvp_engenharia_dados.gold.dim_localidade l
  ON f.ID_MUNICIPIO_RESIDENCIA = l.ID_MUNICIPIO
JOIN mvp_engenharia_dados.gold.dim_causa c
  ON f.ID_CAUSA = c.ID_CID
WHERE l.FLAG_SALVADOR = true
GROUP BY c.CODIGO_CID, c.NOME_CID, c.NOME_CAPITULO
ORDER BY total_obitos DESC
LIMIT 15

In [0]:
%sql
-- Distribuição por sexo
SELECT
  f.SEXO,
  COUNT(*) AS total_obitos
FROM mvp_engenharia_dados.gold.fato_obitos f
JOIN mvp_engenharia_dados.gold.dim_localidade l
  ON f.ID_MUNICIPIO_RESIDENCIA = l.ID_MUNICIPIO
WHERE l.FLAG_SALVADOR = true
GROUP BY f.SEXO
ORDER BY total_obitos DESC

In [0]:
%sql
-- Distribuição por faixa etária
SELECT
  f.FAIXA_ETARIA,
  COUNT(*) AS total_obitos
FROM mvp_engenharia_dados.gold.fato_obitos f
JOIN mvp_engenharia_dados.gold.dim_localidade l
  ON f.ID_MUNICIPIO_RESIDENCIA = l.ID_MUNICIPIO
WHERE l.FLAG_SALVADOR = true
GROUP BY f.FAIXA_ETARIA
ORDER BY total_obitos DESC

In [0]:
%sql
-- Cruzamento faixa etária x sexo
SELECT
  f.FAIXA_ETARIA,
  f.SEXO,
  COUNT(*) AS total_obitos
FROM mvp_engenharia_dados.gold.fato_obitos f
JOIN mvp_engenharia_dados.gold.dim_localidade l
  ON f.ID_MUNICIPIO_RESIDENCIA = l.ID_MUNICIPIO
WHERE l.FLAG_SALVADOR = true
GROUP BY f.FAIXA_ETARIA, f.SEXO
ORDER BY f.FAIXA_ETARIA, f.SEXO

In [0]:
%sql
-- Comparação Salvador x Bahia (total e participação percentual)
SELECT
  CASE WHEN l.FLAG_SALVADOR THEN 'Salvador' ELSE 'Demais municípios da Bahia' END AS localidade,
  COUNT(*) AS total_obitos,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentual
FROM mvp_engenharia_dados.gold.fato_obitos f
JOIN mvp_engenharia_dados.gold.dim_localidade l
  ON f.ID_MUNICIPIO_RESIDENCIA = l.ID_MUNICIPIO
WHERE l.CODIGO_UF = '29'  -- código UF da Bahia
GROUP BY l.FLAG_SALVADOR

In [0]:
%sql
-- Evolução anual: Salvador x Bahia
SELECT
  f.ANO_OBITO,
  CASE WHEN l.FLAG_SALVADOR THEN 'Salvador' ELSE 'Demais municípios da Bahia' END AS localidade,
  COUNT(*) AS total_obitos
FROM mvp_engenharia_dados.gold.fato_obitos f
JOIN mvp_engenharia_dados.gold.dim_localidade l
  ON f.ID_MUNICIPIO_RESIDENCIA = l.ID_MUNICIPIO
WHERE l.CODIGO_UF = '29'
GROUP BY f.ANO_OBITO, l.FLAG_SALVADOR
ORDER BY f.ANO_OBITO, localidade

In [0]:
%sql
-- Evolução anual das 5 principais causas em Salvador
SELECT
  f.ANO_OBITO,
  c.NOME_CID,
  COUNT(*) AS total_obitos
FROM mvp_engenharia_dados.gold.fato_obitos f
JOIN mvp_engenharia_dados.gold.dim_localidade l
  ON f.ID_MUNICIPIO_RESIDENCIA = l.ID_MUNICIPIO
JOIN mvp_engenharia_dados.gold.dim_causa c
  ON f.ID_CAUSA = c.ID_CID
WHERE l.FLAG_SALVADOR = true
  AND c.NOME_CID IN (
    SELECT c2.NOME_CID
    FROM mvp_engenharia_dados.gold.fato_obitos f2
    JOIN mvp_engenharia_dados.gold.dim_localidade l2 ON f2.ID_MUNICIPIO_RESIDENCIA = l2.ID_MUNICIPIO
    JOIN mvp_engenharia_dados.gold.dim_causa c2 ON f2.ID_CAUSA = c2.ID_CID
    WHERE l2.FLAG_SALVADOR = true
    GROUP BY c2.NOME_CID
    ORDER BY COUNT(*) DESC
    LIMIT 5
  )
GROUP BY f.ANO_OBITO, c.NOME_CID
ORDER BY f.ANO_OBITO, total_obitos DESC